---
title: Tracks widget
---

In [2]:
import numpy as np
import pandas as pd
from geneinfo.widget import Tracks
from vscodenb import set_vscode_theme
set_vscode_theme()

Some synthetic data:

In [3]:
#| echo off
from geneinfo.coords import chromosome_lengths
chrom_sizes = dict(chromosome_lengths(assembly='hg38'))
rng = np.random.default_rng(42)
quant_rows = []
n_points = 1000
for chrom, csz in chrom_sizes.items():
    positions = np.sort(rng.integers(0, csz, n_points))
    fst = np.abs(rng.normal(0.05, 0.08, n_points)).clip(0, 1)
    rate = np.cumsum(rng.normal(0, 0.3, n_points))
    rate = (rate - rate.min()) / (rate.max() - rate.min()) * 5 + 0.5
    mid = np.cumsum(rng.normal(0, 0.15, n_points))
    eig = np.sin(positions / 1e7 * 2 * np.pi) + rng.normal(0, 0.2, n_points)
    ci_lo = mid - rng.exponential(0.3, n_points)
    ci_hi = mid + rng.exponential(0.3, n_points) + 10
    depth = rng.poisson(30, n_points).astype(float)

    for pop in ['AFR', 'EUR']:
        noise = rng.normal(0, 0.02, n_points)
        for i in range(n_points):
            quant_rows.append({
                'chrom': chrom, 'pos': int(positions[i]),
                'fst': float(fst[i] + noise[i]),
                'rate': float(rate[i] + noise[i] * 10) + (5 if pop == 'EUR' else 0),
                'eig': float(eig[i] + noise[i]),
                'ci_lo': float(ci_lo[i]) + (15 if pop == 'EUR' else 0), 
                'ci_hi': float(ci_hi[i]) + (15 if pop == 'EUR' else 0), 
                'depth': float(depth[i] + rng.normal(0, 3)),
                'pop': pop,
            })
qdf = pd.DataFrame(quant_rows)            

In [4]:
qdf.head()

,chrom,pos,fst,rate,eig,ci_lo,ci_hi,depth,pop
0,chr1,1090224,0.044858,5.824997,0.811761,-0.455446,10.059748,29.708953,AFR
1,chr1,1217540,0.088215,5.465193,0.749436,-0.165021,10.434385,31.654820,AFR
2,chr1,1321229,0.038109,5.644535,0.775922,0.183732,10.394984,26.824892,AFR
3,chr1,1351792,0.063331,5.054980,0.663487,0.311840,10.397019,32.699465,AFR
4,chr1,1620963,0.126832,5.321591,0.598848,0.114335,11.571794,19.297649,AFR


In [5]:
viewer1 = (
    Tracks(chrom_sizes)
    .add_scatter_track(
        qdf, 'Scatter', x='pos', y='fst', group_by='pop', 
        tip_fmt=False,
    )
    .add_line_track(
        qdf, 'Lines', x='pos', y='rate', group_by='pop',
        step='post', tip_fmt=False, height=60,
    )
    .add_fill_track(
        qdf, 'Fill btwn', x='pos', y_hi='ci_hi', y_lo='ci_lo', 
        group_by='pop', height=60, tip_fmt=False,
    )
    .add_fill_track(
        qdf, 'Fill baseln', x='pos', y='eig', group_by='pop', 
        step='post', height=60, tip_fmt=False,
    )
    .add_histogram_track(
        qdf, 'Histogram', x='pos', y='depth', group_by='pop',
        stack=True,
        height=60, tip_fmt=False, 
    )
    .add_gene_track(assembly='hg38', height = 100)
)
# viewer1.zoom_to('chr1', center=70000000, window=1200000)
viewer1

## Highlighting

In [6]:
viewer2 = (
    Tracks(chrom_sizes)
    .add_gene_track(assembly='hg38',
                    height = 100, 
                    highlight=['LRRC7', 'LRRC40', 'CTH', 'SRSF11', 'CTH'], 
                    highlight_color="#ee55c5",
                    )
)
# zoom to region interest for demonstration purposes
viewer2.zoom_to('chr1', center=70000000, window=1200000)

In [7]:
viewer3 = (
    Tracks(chrom_sizes)
    .add_gene_track(assembly='hg38',
                    height = 100, 
                    highlight={                                                                   
                        'fill':      ['ANKRD13C', 'LRRC40', 'CTH'],                                           
                        'stroke':    ['LRRC40'],                                                     
                        'outline':   ['BRCA1'],                                                   
                        'color':     ['LRRC40'],                                                    
                        'bold':      ['LRRC7', 'LRRC40'],                                           
                        'italic':    ['ANKRD13C'],                                                     
                        'underline': ['CTH'],                                                    
                        'halo':      ['SRSF11'],                                                   
                    }, 
                    highlight_fill_color    = "#e03a4e",                  
                    highlight_spine_color   = "#d80ce6",                                          
                    highlight_outline_color = "#000000",                                          
                    highlight_label_color   = "#169f4a",                                          
                    highlight_halo_color    = "#79CAD32E",    
                    )
)
# zoom to region interest for demonstration purposes
viewer3.zoom_to('chr1', center=70300000, window=600000)

## Segments

Synthetic segment data:

In [8]:
#| echo off
df_list = []
for chrom, csz in chrom_sizes.items():
    for group in ['EUR', 'EAS']:
        for ind in [f'{group}_ind{i}' for i in range(1, 11)]:
            for kind in ['Neanderthal', 'Denisovan']:
                n_points = 100
                positions = np.sort(rng.integers(0, csz, n_points))
                length = rng.integers(10000, 1000000, n_points)
                df = pd.DataFrame({
                    'chrom': chrom,
                    'start': positions,
                    'end': positions + length,
                    'ind': ind,
                    'reg': group,
                    'ancestry': kind,
                })
                df_list.append(df)
segs = pd.concat(df_list, ignore_index=True)

hm_df = segs[['chrom', 'start', 'end', 'ind', 'reg']].rename(
    columns={'ind': 'sample', 'reg': 'pop'}
)

In [9]:
segs.head()

,chrom,start,end,ind,reg,ancestry
0,chr1,9264851,9642214,EUR_ind1,EUR,Neanderthal
1,chr1,14200255,15198134,EUR_ind1,EUR,Neanderthal
2,chr1,15108496,15182094,EUR_ind1,EUR,Neanderthal
3,chr1,15970719,16967259,EUR_ind1,EUR,Neanderthal
4,chr1,22242645,22531349,EUR_ind1,EUR,Neanderthal


In [10]:
hm_df.head()

,chrom,start,end,sample,pop
0,chr1,9264851,9642214,EUR_ind1,EUR
1,chr1,14200255,15198134,EUR_ind1,EUR
2,chr1,15108496,15182094,EUR_ind1,EUR
3,chr1,15970719,16967259,EUR_ind1,EUR
4,chr1,22242645,22531349,EUR_ind1,EUR


In [11]:


viewer4 = (
    Tracks(chrom_sizes)
    .add_segment_track(
        segs.loc[segs['reg'] == 'EUR'], 'EUR', group_by='ancestry', individual_col='ind',
        tip_fmt=False, stack=True,
        height=60, density_windows=(512, 2048, 8192),
    )   
    .add_segment_track(
        segs.loc[segs['reg'] == 'EAS'], 'EAS', group_by='ancestry', individual_col='ind',
          tip_fmt=False, stack=True,
        height=60, density_windows=(512, 2048, 8192),
    )    
    .add_heatmap_track(
        segs, 'Individuals',
        individual_col='ind',
            group_col='reg',
        height=390,   # will be expanded to accommodate on px per segment row
        windows=5000, # max is 8192 or 16384 depending on GPU
        tip_fmt=False,
    )    
    .add_gene_track(assembly='hg38', height = 100)
)
viewer4

## Tooltip

Each track contributes to the cursor tooltip unless `tip_fmt=False`. Use `tip_fmt` to control what each track contributes to the tooltip. Format placeholders use Python-style `{key}` or `{key:.Nf}` syntax.

Available keys per track type:
- **Gene**: `name`, `strand`, `start`, `end`
- **Scatter / Line**: `group`, `value`, `x`
- **Fill**: `group`, `lo`, `hi`, `x`
- **Histogram**: `group`, `value`, `x`
- **Segment**: `group`
- **Heatmap**: `group`, `individual`, `nInd`

If a format string references an invalid key, the tooltip shows the available keys as a hint.

In [21]:
viewer5 = (
    Tracks(chrom_sizes)
    .add_scatter_track(
        qdf, 'Scatter', x='pos', y='fst', group_by='pop',
        tip_fmt="{value:.4f} is the value for {group}",
    )
    .add_line_track(
        qdf, 'Lines', x='pos', y='rate', group_by='pop',
        tip_fmt="For {group} the value is {value:.2f}",
    )
    .add_gene_track(
        assembly='hg38', height=44, 
        tip_fmt="The gene at this pos is {name}",
    )
)
viewer5

## Theme

The widget has a light and dark default theme determined by the vscodenb package if installed. You get the active theme defaults like this:

In [ ]:
from geneinfo.widget import get_default_theme, set_default_theme
theme = get_default_theme()
theme

{'bg': '#FAF9F6',
 'fg': '#1a1a2e',
 'panel': '#FAF9F6',
 'border': '#FAF9F6',
 'input_bg': '#ffffff',
 'input_fg': '#1a1a2e',
 'input_border': '#c0c0cc',
 'accent': '#3355dd',
 'muted': '#888899',
 'label': '#555566',
 'gene_exon': '#4488cc',
 'gene_spine': '#666688',
 'gene_label': '#9090c0',
 'highlight_fill': '#e03a4e',
 'highlight_spine': '#d80ce6',
 'highlight_outline': '#000000',
 'highlight_label': '#169f4a',
 'highlight_halo': '#4152CF2F'}

You can change the theme colors and save them back for use in subsequent viewers:

In [ ]:
# theme['input_bg'] = 'pink'
set_default_theme(theme)